# Table 1: AoU-LR cohort summary

Rebuild manuscript Table 1 from the person-level covariates table, with a few
tech-specific supplements for fields not yet first-class in the covariates export.

**Primary input**

- `covariates.source_rebuilt.csv.gz`

**Supplements** (only where covariates lack tech-specific values)

- `resources/legacy_covariates/Integratedcall-GRCh38.tsv` — Phase 1 PacBio median read length
- `resources/legacy_covariates/merged_all_df.csv.gz` — Phase 2 PacBio/ONT platform-specific coverage and read length
- `resources/legacy_covariates/ont-sample-hg38.tsv` — Phase 1 ONT coverage / read length (also in covariates as `ont_*`)
- `resources/legacy_covariates/aou_phase2.ped` — pedigree structures

**Stratum definitions**

| Column | Definition |
|---|---|
| Phase 1 mid-pass | `in_cdr_v7` (n = 1,027) |
| Phase 2 mid-pass | final-releasable PacBio discovery samples not in high-pass |
| Phase 2 high-pass | all final-releasable PacBio+ONT dual-tech samples, plus highest-coverage PacBio-only samples to n = 1,133 |

This high-pass proxy recovers the manuscript pattern of ONT overlap concentrated in
the high-pass column. An explicit mid/high-pass cohort label is not yet stored in
the covariates table.

**Outputs**

- `summaries/manuscript/table1_cohort_summary.tsv`
- `summaries/manuscript/table1_cohort_summary.md`



In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Bootstrap scripts/ from $WORKSPACE_BUCKET/scripts/ when not on the VM.
for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError(
            "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
            "Upload scripts/ to gs://WORKSPACE/scripts/."
        )
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        ["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"]
    )
    sys.path.insert(0, str(_dest))

from __future__ import annotations

from terra_notebook import init_notebook

SCRIPTS = init_notebook("workspace_paths.py")
from workspace_paths import data_root

import numpy as np
import pandas as pd

ROOT = data_root()
RESOURCES = ROOT / "resources"
LEGACY = RESOURCES / "legacy_covariates"

COV_CSV = ROOT / "covariates.source_rebuilt.csv.gz"
INTEGRATEDCALL_TSV = LEGACY / "Integratedcall-GRCh38.tsv"
MERGED_ALL_CSV = LEGACY / "merged_all_df.csv.gz"
ONT_PHASE1_TSV = LEGACY / "ont-sample-hg38.tsv"
PEDIGREE_CSV = LEGACY / "aou_phase2.ped"

OUT_DIR = ROOT / "summaries" / "manuscript"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_TSV = OUT_DIR / "table1_cohort_summary.tsv"
OUT_MD = OUT_DIR / "table1_cohort_summary.md"

HIGH_PASS_N = 1133

for path in [COV_CSV, INTEGRATEDCALL_TSV, MERGED_ALL_CSV, ONT_PHASE1_TSV, PEDIGREE_CSV]:
    assert path.exists(), path

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)


In [ ]:
def mean_sd(series: pd.Series, digits: int = 1) -> str:
    x = pd.to_numeric(series, errors="coerce").dropna()
    if len(x) == 0:
        return "—"
    if len(x) == 1:
        return f"{x.iloc[0]:.{digits}f}"
    return f"{x.mean():.{digits}f} ± {x.std(ddof=1):.{digits}f}"


def mean_sd_int(series: pd.Series) -> str:
    x = pd.to_numeric(series, errors="coerce").dropna()
    if len(x) == 0:
        return "—"
    if len(x) == 1:
        return f"{x.iloc[0]:.0f}"
    return f"{x.mean():.0f} ± {x.std(ddof=1):.0f}"


def overlap_label(n_overlap: int, n_total: int) -> str:
    if n_total == 0:
        return "—"
    if n_overlap == 0:
        return "0"
    return f"{n_overlap}/{n_total:,}"



In [ ]:
cov = pd.read_csv(
    COV_CSV,
    dtype={"research_id": str, "biobank_id": str},
    low_memory=False,
)

integrated = pd.read_csv(
    INTEGRATEDCALL_TSV,
    sep="\t",
    dtype={"entity:Integratedcall-GRCh38_id": str},
).rename(columns={"entity:Integratedcall-GRCh38_id": "research_id"})
integrated["aligned_read_length_median"] = pd.to_numeric(
    integrated["aligned_read_length_median"], errors="coerce"
)

merged = pd.read_csv(MERGED_ALL_CSV, dtype={"person_id": str}, low_memory=False)

pb_merged = (
    merged.loc[merged["platform"].isin(["Revio", "Sequel"])]
    .sort_values(["person_id", "coverage", "rl_median"], ascending=[True, False, False])
    .drop_duplicates("person_id", keep="first")
    .rename(columns={
        "person_id": "research_id",
        "coverage": "pb_coverage",
        "rl_median": "pb_read_length_median",
        "platform": "pb_platform",
    })[
        ["research_id", "pb_coverage", "pb_read_length_median", "pb_platform"]
    ]
)

ont_merged = (
    merged.loc[merged["platform"].astype(str).str.startswith("ONT", na=False)]
    .sort_values(["person_id", "coverage", "rl_median"], ascending=[True, False, False])
    .drop_duplicates("person_id", keep="first")
    .rename(columns={
        "person_id": "research_id",
        "coverage": "ont_coverage_merged",
        "rl_median": "ont_read_length_median_merged",
        "platform": "ont_platform_merged",
    })[
        ["research_id", "ont_coverage_merged", "ont_read_length_median_merged", "ont_platform_merged"]
    ]
)

ont_phase1 = pd.read_csv(
    ONT_PHASE1_TSV,
    sep="\t",
    dtype={"entity:ont-sample-hg38_id": str},
).rename(columns={"entity:ont-sample-hg38_id": "research_id"})
ont_phase1_ids = set(ont_phase1["research_id"])

ped = pd.read_csv(PEDIGREE_CSV, dtype=str)

print(f"covariates: {len(cov):,} people")
print(f"PacBio merged metrics: {len(pb_merged):,}")
print(f"ONT merged metrics: {len(ont_merged):,}")
print(f"Phase 1 ONT sample sheet: {len(ont_phase1_ids):,}")
print(f"pedigree rows: {len(ped):,}")



## Define Table 1 strata

Phase 2 high-pass is not an explicit covariates field. The proxy below matches the
manuscript pattern: ONT overlap sits in high-pass, mid-pass PacBio discovery has
no ONT overlap, and high-pass PacBio discovery size is 1,133.



In [ ]:
phase1 = cov.loc[cov["in_cdr_v7"]].copy()
phase1 = phase1.merge(
    integrated[["research_id", "aligned_read_length_median"]],
    on="research_id",
    how="left",
    validate="one_to_one",
)

pacbio = cov.loc[cov["final_releasable_v9"] & cov["technology"].eq("PacBio")].copy()
pacbio = pacbio.merge(pb_merged, on="research_id", how="left", validate="one_to_one")
assert pacbio["pb_coverage"].notna().all(), "missing PacBio-specific coverage for releasable PacBio discovery samples"

dual = pacbio.loc[pacbio["has_ONT"]].copy()
pacbio_only = pacbio.loc[~pacbio["has_ONT"]].copy()
n_fill = HIGH_PASS_N - len(dual)
assert n_fill >= 0, (HIGH_PASS_N, len(dual))
high = pd.concat(
    [dual, pacbio_only.nlargest(n_fill, "pb_coverage")],
    ignore_index=True,
)
assert len(high) == HIGH_PASS_N
assert high["research_id"].is_unique

mid = pacbio.loc[~pacbio["research_id"].isin(set(high["research_id"]))].copy()
assert len(mid) + len(high) == len(pacbio)
assert int(mid["has_ONT"].sum()) == 0

strata = {
    "phase1_midpass": phase1,
    "phase2_midpass": mid,
    "phase2_highpass": high,
}

print({k: len(v) for k, v in strata.items()})
print("high-pass ONT dual-tech:", int(high["has_ONT"].sum()))
print(
    "ONT-primary releasable samples not shown in PacBio-centric columns:",
    int((cov["final_releasable_v9"] & cov["technology"].eq("ONT")).sum()),
)



In [ ]:
def classify_families(pedigree: pd.DataFrame) -> list[tuple[str, set[str]]]:
    families: list[tuple[str, set[str]]] = []
    for _, group in pedigree.groupby("ped"):
        ids = set(group["id"])
        n_founders = int(((group["father"].eq("0")) & (group["mother"].eq("0"))).sum())
        n_kids = len(group) - n_founders
        multigen = False
        for _, row in group.iterrows():
            if row["father"] == "0" and row["mother"] == "0":
                continue
            for parent in (row["father"], row["mother"]):
                if parent not in ids:
                    continue
                pref = group.loc[group["id"].eq(parent)].iloc[0]
                if not (pref["father"] == "0" and pref["mother"] == "0"):
                    multigen = True
        if multigen:
            kind = "multi-generational"
        elif len(group) == 3 and n_founders == 2 and n_kids == 1:
            kind = "trio"
        elif len(group) == 4 and n_founders == 2 and n_kids == 2:
            kind = "quartet"
        else:
            kind = "other"
        families.append((kind, ids))
    return families


FAMILIES = classify_families(ped)


def pedigree_counts(id_set: set[str]) -> dict[str, int]:
    counts = {"trio": 0, "quartet": 0, "multi-generational": 0}
    for kind, ids in FAMILIES:
        if kind in counts and ids <= id_set:
            counts[kind] += 1
    return counts


def ont_overlap_frame(stratum: pd.DataFrame, *, phase1: bool) -> pd.DataFrame:
    """ONT metrics for people in the stratum who have ONT data."""
    ids = set(stratum["research_id"])

    if phase1:
        # Manuscript Phase 1 ONT overlap is the curated 50-sample sheet.
        hit = ids & ont_phase1_ids
        return (
            cov.loc[cov["research_id"].isin(hit), [
                "research_id", "ont_coverage", "ont_read_length_median"
            ]]
            .rename(columns={
                "ont_coverage": "ont_coverage_use",
                "ont_read_length_median": "ont_rl_use",
            })[
                ["research_id", "ont_coverage_use", "ont_rl_use"]
            ]
            .reset_index(drop=True)
        )

    # Phase 2: prefer covariates ont_* when present, else merged platform rows.
    rows: list[dict[str, object]] = []
    cov_ont = cov.loc[
        cov["research_id"].isin(ids) & cov["ont_coverage"].notna(),
        ["research_id", "ont_coverage", "ont_read_length_median"],
    ]
    for _, row in cov_ont.iterrows():
        rows.append({
            "research_id": row["research_id"],
            "ont_coverage_use": row["ont_coverage"],
            "ont_rl_use": row["ont_read_length_median"],
        })
    seen = {r["research_id"] for r in rows}
    merged_ont = ont_merged.loc[
        ont_merged["research_id"].isin(ids - seen),
        ["research_id", "ont_coverage_merged", "ont_read_length_median_merged"],
    ]
    for _, row in merged_ont.iterrows():
        rows.append({
            "research_id": row["research_id"],
            "ont_coverage_use": row["ont_coverage_merged"],
            "ont_rl_use": row["ont_read_length_median_merged"],
        })
    return pd.DataFrame(rows, columns=["research_id", "ont_coverage_use", "ont_rl_use"])


def stratum_column(stratum: pd.DataFrame, *, phase1: bool) -> dict[str, object]:
    ids = set(stratum["research_id"])
    ont = ont_overlap_frame(stratum, phase1=phase1)
    ped_counts = pedigree_counts(ids)

    if phase1:
        pb_coverage = stratum["coverage"]
        pb_rl = stratum["aligned_read_length_median"]
    else:
        pb_coverage = stratum["pb_coverage"]
        pb_rl = stratum["pb_read_length_median"]

    return {
        "PacBio discovery participants": f"{len(stratum):,}",
        "PacBio mean coverage": mean_sd(pb_coverage),
        "PacBio median read length (bp)": mean_sd_int(pb_rl),
        "ONT overlapping participants": overlap_label(len(ont), len(stratum)),
        "ONT mean coverage": mean_sd(ont["ont_coverage_use"]) if len(ont) else "—",
        "ONT median read length (bp)": mean_sd_int(ont["ont_rl_use"]) if len(ont) else "—",
        "Pedigrees: trios": ped_counts["trio"],
        "Pedigrees: quartets": ped_counts["quartet"],
        "Pedigrees: multi-generational": ped_counts["multi-generational"],
        "Participants with CpG methylation": int(stratum["has_methylation"].sum()),
        "Participants with RNA-seq": int(stratum["has_rna"].sum()),
        "Participants with Olink proteomics": int(stratum["has_proteomics"].sum()),
        "Participants with EHRs": int(stratum["has_ehr_data"].fillna(False).sum()),
    }



In [ ]:
table = pd.DataFrame({
    "AoU-LR Phase 1: mid-pass (~8x)": stratum_column(strata["phase1_midpass"], phase1=True),
    "AoU-LR Phase 2: mid-pass (~15x)": stratum_column(strata["phase2_midpass"], phase1=False),
    "AoU-LR Phase 2: high-pass (~30x)": stratum_column(strata["phase2_highpass"], phase1=False),
})
table.index.name = "Metric"
table = table.reset_index()

# Section labels matching the manuscript layout
section_for = {
    "PacBio discovery participants": "Sequencing platform — PacBio",
    "PacBio mean coverage": "Sequencing platform — PacBio",
    "PacBio median read length (bp)": "Sequencing platform — PacBio",
    "ONT overlapping participants": "Sequencing platform — ONT",
    "ONT mean coverage": "Sequencing platform — ONT",
    "ONT median read length (bp)": "Sequencing platform — ONT",
    "Pedigrees: trios": "Relatedness",
    "Pedigrees: quartets": "Relatedness",
    "Pedigrees: multi-generational": "Relatedness",
    "Participants with CpG methylation": "Multi-omic and EHR linkage",
    "Participants with RNA-seq": "Multi-omic and EHR linkage",
    "Participants with Olink proteomics": "Multi-omic and EHR linkage",
    "Participants with EHRs": "Multi-omic and EHR linkage",
}
table.insert(0, "Section", table["Metric"].map(section_for))

try:
    display(table)
except NameError:
    print(table.to_string(index=False))

table.to_csv(OUT_TSV, sep="\t", index=False)

md_lines = [
    "# Table 1. AoU-LR cohort summary",
    "",
    "Generated from `covariates.source_rebuilt.csv.gz` with tech-specific supplements.",
    "",
    "| Section | Metric | Phase 1 mid-pass (~8x) | Phase 2 mid-pass (~15x) | Phase 2 high-pass (~30x) |",
    "|---|---|---:|---:|---:|",
]
for _, row in table.iterrows():
    md_lines.append(
        f"| {row['Section']} | {row['Metric']} | "
        f"{row['AoU-LR Phase 1: mid-pass (~8x)']} | "
        f"{row['AoU-LR Phase 2: mid-pass (~15x)']} | "
        f"{row['AoU-LR Phase 2: high-pass (~30x)']} |"
    )
md_lines.extend([
    "",
    "## Definitions",
    "",
    "- Phase 1: `in_cdr_v7`.",
    "- Phase 2 PacBio discovery: `final_releasable_v9` and `technology == PacBio`.",
    f"- Phase 2 high-pass: all PacBio+ONT dual-tech discovery samples, plus highest-coverage PacBio-only samples to n = {HIGH_PASS_N:,}.",
    "- Phase 2 mid-pass: remaining PacBio discovery samples.",
    "- Pedigrees counted only when every family member falls in the stratum.",
    "- ONT-primary releasable samples are not shown in this PacBio-centric layout.",
    "",
])
OUT_MD.write_text("\n".join(md_lines))
print("wrote", OUT_TSV)
print("wrote", OUT_MD)

